In [0]:
# Step 1: Read the raw JSON file from the landing zone. The file is multi-line and contains nested structures.
from pyspark.sql import functions as F

raw_path = '/Volumes/workspace/landingzone/energy_transfer/profisee_energy_transfer.json'

# Read the multi-line JSON file and infer schema. This creates a DataFrame with a single row containing exportMetadata and members.
raw_df = (
    spark.read.format('json')
    .option('multiLine', True)          # Required for multi-line JSON
    .option('inferSchema', True)
    .load(raw_path)
)

# Print the schema to inspect the structure of the loaded data.
raw_df.printSchema()

# Step 2: Explode the 'members' array so each member becomes a row. Extract relevant fields and flatten nested auditInfo.
bronze = (
    raw_df
    .select(
        # Metadata columns
        F.col('exportMetadata.source').alias('source_system'),
        F.col('exportMetadata.tenant').alias('tenant'),
        F.col('exportMetadata.domain').alias('domain'),
        F.col('exportMetadata.exportDate').alias('export_date'),
        # Explode nested array → one row per member
        F.explode('members').alias('member')
    )
    .select(
        'source_system', 'tenant', 'domain', 'export_date',
        F.col('member.memberId').alias('member_id'),
        F.col('member.memberCode').alias('member_code'),
        F.col('member.entity').alias('entity'),
        F.col('member.memberType').alias('member_type'),
        F.col('member.matchStatus').alias('match_status'),
        F.col('member.matchGroupId').alias('match_group_id'),
        # Keep the full attributes struct for downstream Silver parsing
        F.col('member.attributes').alias('attributes'),
        # Flatten audit info
        F.col('member.auditInfo.createdBy').alias('created_by'),
        F.col('member.auditInfo.createdOn').cast('timestamp').alias('created_on'),
        F.col('member.auditInfo.updatedBy').alias('updated_by'),
        F.col('member.auditInfo.updatedOn').cast('timestamp').alias('updated_on'),
        F.col('member.auditInfo.version').cast('int').alias('record_version'),
        F.current_timestamp().alias('_ingested_at')
    )
)

# Display the transformed DataFrame for visual inspection.
display(bronze)

# Step 3: Save the processed data as a Delta table for downstream analysis and reporting.
bronze.write.format('delta').mode('overwrite').saveAsTable('energy_transfer_profisee')

print(f"✅ Saved {bronze.count()} records to energy_transfer_profisee")

In [0]:
# Load the Delta table created in the previous step into a DataFrame for analysis.
df = spark.table('energy_transfer_profisee')
# Display the DataFrame to review the ingested and transformed data.
display(df)

In [0]:
# Generate summary statistics (count, mean, stddev, min, max) for all columns in the DataFrame.
df.summary().show()

In [0]:
# Print the schema of the DataFrame to understand column types and structure.
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Show descriptive statistics for numeric columns (count, mean, stddev, min, max).
df.describe().display()

# Calculate the number of nulls in each column to assess data quality.
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

# Calculate the number of distinct values in each column to understand cardinality.
df.select([F.countDistinct(c).alias(c) for c in df.columns]).display()

In [0]:
# Display the DataFrame and use the Databricks Data Profile tab for interactive column profiling.
display(df)
# Then click the "Data Profile" tab in the output widget

In [0]:
# If you need the full summarize functionality, switch to a classic (non-serverless) compute cluster.
# dbutils.data.summarize() works fine there.
#dbutils.data.summarize()

In [0]:
from pyspark.sql import functions as F

# Define a function to profile the DataFrame: returns null counts, distinct counts, and percentages for each column.
def profile_df(df):
    rows = df.count()
    stats = []
    for c in df.columns:
        col_type = dict(df.dtypes)[c]
        null_count = df.where(F.col(c).isNull()).count()
        distinct_count = df.select(F.countDistinct(c)).collect()[0][0]
        stats.append({
            "column": c,
            "type": col_type,
            "nulls": null_count,
            "null_pct": round(null_count / rows * 100, 2),
            "distinct": distinct_count,
            "distinct_pct": round(distinct_count / rows * 100, 2)
        })
    return spark.createDataFrame(stats)

# Run the profiling function and display the results.
profile_df(df).display()